# Main notebook for the biome project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
import shap
import seaborn as sns
import polars as pl

/Users/andrearaaschou/courses/BERN01/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Data

In [ ]:
# Load data 
lpj_df = pd.read_csv("../data/LPJ-GUESS_output_BERN1.csv")
grid_df = pd.read_csv("../data/gridlist_pan_gfed_ISO3_UN_ecoregion.txt",sep=" ")
soil_df = pd.read_csv("../data/soilmap.dat",sep="\t")

# Load all climate datafiles and put in one dictionary
climate_files = {
    "tswrf": "../data/Tswrfdaymean1961_1990.csv",
    "pre": "../data/Predaymean1961_1990.csv",
    "tmp": "../data/Tmpdaymean1961_1990.csv",
    "tmax": "../data/Tmaxdaymean1961_1990.csv",
    "tmin": "../data/Tmindaymean1961_1990.csv"}
climate_df = {}
for name, path in climate_files.items():
    climate_df[name] = pd.read_csv(path)

# Load all climate statistics datafiles and put in one dictionary
stats_files = {
    "tswrf": "../data/Tswrfdaymean1961_1990_stats.csv",
    "pre": "../data/Predaymean1961_1990_stats.csv",
    "tmp": "../data/Tmpdaymean1961_1990_stats.csv",
    "tmax": "../data/Tmaxdaymean1961_1990_stats.csv",
    "tmin": "../data/Tmindaymean1961_1990_stats.csv"}
stats_df = {}
for name, path in stats_files.items():
    stats_df[name] = pd.read_csv(path)

In [3]:
# Fix problems with soil_df
soil_df = soil_df.dropna() # drops all rows with nan values
soil_df[soil_df["lon"].isna()]
soil_df["lon"] = pd.to_numeric(soil_df["lon"]) # Change the lon column values from type str to float
soil_df.shape # check how many rows were dropped
soil_df = soil_df.rename(columns={"lon": "Lon","lat": "Lat"}) # Rename columns

In [5]:
# Merge df to one collected_df
collected_df = lpj_df[["Lon", "Lat", "Biome_obs"]].copy() # Start by taking wanted columns from lpj_df
collected_df = collected_df.merge(soil_df, on=["Lon", "Lat"], how="left") # Merge with soil_df
for name, df in stats_df.items(): # Merge with stats (climate) as well
    collected_df = collected_df.merge(
        df,
        on=["Lon", "Lat"],
        how="left",
        suffixes=("", f"_{name}") # add suffixes to keep climate columns apart
    )

collected_df.head()

,Lon,Lat,Biome_obs,clay,silt,sand,orgC,CN,pH,cellfraction,...,SpringStd_tmin,SummerMean_tmin,SummerMedian_tmin,SummerStd_tmin,FallMean_tmin,FallMedian_tmin,FallStd_tmin,WinterMean_tmin,WinterMedian_tmin,WinterStd_tmin
0,39.75,-1.25,12,0.19,0.16,0.65,0.007,12.0,6.2,0.663,...,0.85947,297.03,297.02,0.77132,295.42,295.38,0.19342,296.47,296.48,0.32203
1,150.25,-34.25,6,0.17,0.22,0.61,0.005,12.0,6.2,0.405,...,1.23520,280.58,280.14,2.60160,278.59,277.94,2.10780,286.79,287.04,2.03760
2,-63.75,82.75,17,0.08,0.37,0.55,0.020,11.0,5.9,1.000,...,1.13850,259.44,260.53,10.01100,268.84,272.09,5.76450,246.56,244.91,4.44050
3,59.25,30.75,14,0.19,0.26,0.55,0.007,10.0,7.9,0.383,...,3.41820,297.17,298.07,3.95480,299.81,300.63,2.58780,285.18,283.81,5.08320
4,24.25,27.75,17,0.18,0.28,0.54,0.004,14.0,8.1,0.487,...,2.18730,296.19,296.15,2.91960,298.87,299.31,1.03790,289.35,289.10,4.03760


## 3. Binary classification

In this part, you are going to build a binary classifier to classify two classes of biomes from
“Biome_obs”. You can choose one of the biomes and try to train a classifier that can distinguish this biome from the other data, e.g, ’Arctic/alpine tundra’ or ’not Arctic/alpine tundra’.

One way to narrow down the data if you have trouble creating the binary classifier is to
only include samples from two different country codes or regions

In [ ]:
# Make binary classes (desert/not desert)
# Use the column Biome_obs in lpj_df for each location to get the actual outcome (y)
# Use climate and soil type as predictors for biome type (x_i?)
# Already prepared one collected df to use

# Train on one country, predict for one country or randomly sample points from one country and predict for the same country



### 3.1. Data split

Split your dataset into two parts: a training set and a test set. The training set is used to
train the model, and the test set is used to evaluate model performance. Think through
how you split your data and what fits your purpose. Ways to split your data could be
through climate, zone, biome, countries or continents. Are all the biomes represented in
both your train and test data? Are the frequency of biome classes representative?

### 3.2. Model training
Next, train a Random Forest binary classification model. You can start by using training
data from one smaller region (could be one country code). Evaluate your model and try to
optimise the hyperparameters.

### 3.3. Model evaluation
Evaluate the model's performance using appropriate metrics for binary classification. Common metrics include precision, recall, and to plot the predictions in a confusion matrix. Test the model trained on one part of the world on the same biome in another part of the world,
e.g., Arctic/alpine tundra in either North America or Siberia. Does the model show equal 
skill in both areas? Could the features you have chosen affect the skill of the model in another part of the world? Why?

### 3.4. Feature importance and model interpretation
Analyze which features played the most crucial role in your model's decisions. Finally, discuss and interpret your results comprehensively, and explore any misclassifications to gain valuable insights into how well machine learning models adapt to different continents and
environments. Are all features that you used for training necessary? Could you reduce the
number of features used? Were there features that you did not use that could be important? Why would they be important?
Optional: Plot the misclassified regions on a world map.